# C-14 Disaggregated Performance Analysis

**Control:** C-14 Disaggregated Performance Analysis, MAK-AI Responsible AI Toolkit.

**Learning goal:** report how a model performs for each group and each intersection of groups, with the uncertainty that comes with it; avoid over-reading small groups; and notice when the overall number hides what happens inside groups.

**Time to run:** about 4 minutes on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:** Diabetes 130-US Hospitals for Years 1999-2008 (Strack and others, 2014), DOI 10.24432/C5230J, licence CC BY 4.0, 101,766 hospital stays, from https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008. It is credited here because CC BY 4.0 asks for attribution.

**A caveat before you start:** these are stays in United States hospitals. The groups (age band, gender, a handful of US race categories) are the ones the hospitals recorded, and they do not describe your patients or your setting. The data has real missing values. Step 1 lists the most-missing columns (weight is missing for most stays), and people whose race was not recorded are reported as a group called `missing`. In two test-result columns, `max_glu_serum` and `A1Cresult`, the file holds the text `None` where no result was recorded; pandas reads that text as missing, so Step 1 counts those as missing too. The notebook predicts readmission within 30 days, a setting where a missed case has a cost. It teaches a method and says nothing about any real hospital or about your model.

## Linked risk areas and questions

Generated from content 5.1.0. Do not edit by hand.

Risk areas: 2

- IP-01: Algorithmic Discrimination
- IP-02: Aggregation Bias

Questions: 8: Q-PP-23, Q-IP-02, Q-IP-04, Q-IP-06, Q-IP-13, Q-IP-14, Q-IP-15, Q-IP-19

## When to use this notebook

Use it when you are answering a question about how well the model works for different groups and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-IP-04`: have you calculated performance metrics separately for each relevant subgroup?
- `Q-IP-14`: does aggregating data across populations hide differences between subgroups?
- `Q-IP-19`: do the model's errors have a different impact on different groups?

The per-group table this notebook writes is the one a team types into the model card's disaggregated results table.

You need a table with one row per person or case, the true outcome, and either a model's predicted probabilities or the columns to train a simple model.

In [ ]:
import hashlib
import importlib.metadata
import io
import os
import platform
import tempfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fairlearn.metrics import MetricFrame, count, false_positive_rate, selection_rate
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import resample

SEED = 20261030  # the one random seed; every random step below uses it
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy', 'scikit-learn', 'fairlearn', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/296/data.csv'
DATA_SHA256 = 'f792c388d9b470aac4fbf21c39b176534d55a116e7626881c30bbe7ab0341422'
DATA_FILE = 'uci-diabetes-130-296-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


MAX_DOWNLOAD_BYTES = 64 * 1024 * 1024  # a dataset file is never larger than this; a longer body is refused


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. The body is read in 1 MiB pieces and refused past MAX_DOWNLOAD_BYTES.
    `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    if Path(file_name).name != file_name:
        raise ValueError(f'the cache file name must be a plain name: {file_name}')

    def read_checked(stream):
        digest, pieces, size = hashlib.sha256(), [], 0
        for piece in iter(lambda: stream.read(1 << 20), b''):
            size += len(piece)
            if size > MAX_DOWNLOAD_BYTES:
                raise RuntimeError(f'{file_name}: the file is larger than {MAX_DOWNLOAD_BYTES:,} bytes; refused')
            digest.update(piece)
            pieces.append(piece)
        return b''.join(pieces), digest.hexdigest()

    path = CACHE_DIR / file_name
    if path.exists():
        with path.open('rb') as stream:
            data, found = read_checked(stream)
        source, remedy = f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data, found = read_checked(response)
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = tempfile.NamedTemporaryFile(dir=CACHE_DIR, suffix='.part', delete=False)
        try:
            with partial:
                partial.write(data)
            os.replace(partial.name, path)  # the cache never holds a half-written file, whoever else is writing
        except BaseException:
            Path(partial.name).unlink(missing_ok=True)
            raise
    return data


# To use your own file, replace the next line with:  raw = Path('your-file.csv').read_bytes()
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw), low_memory=False)
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
ID_COLUMN = 'patient_nbr'            # one person can appear many times; only the first row per person is kept
ORDER_COLUMN = 'encounter_id'        # the first row is the one with the lowest value in this column
LABEL = 'readmitted'                 # the column that holds the true outcome
POSITIVE = '<30'                     # the value of LABEL that is the case a miss would cost us (readmitted within 30 days)
EXCLUDE_COLUMN = 'discharge_disposition_id'
EXCLUDE_VALUES = [11, 13, 14, 19, 20, 21]   # people who died or went to hospice cannot be readmitted, so they are left out
GROUP_COLUMNS = ['age', 'gender', 'race']   # the columns that define the groups you report on
INTERSECTIONS = [['gender', 'race'], ['age', 'gender']]   # pairs of columns to report together
NUMERIC = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
           'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
CATEGORICAL = ['admission_type_id', 'admission_source_id', 'discharge_disposition_id',
               'insulin', 'change', 'diabetesMed', 'A1Cresult', 'max_glu_serum']
SCORE_COLUMN = None                  # name a column of your own model's scores (0 to 1) to skip the practice model
THRESHOLD = 0.12                     # a score at or above this is flagged; a low threshold catches more cases and flags more people
MIN_GROUP_SIZE = 300                 # a group needs at least this many people in the test set to be reported
MIN_POSITIVES = 30                   # and at least this many true cases, because recall is measured on the cases only
N_BOOTSTRAP = 200                    # resamples for the intervals
FACTOR_COLUMN = 'change'             # a yes/no factor whose link to the outcome you want to check inside groups
FACTOR_VALUE = 'Ch'                  # the value of FACTOR_COLUMN that means "yes"

### Step 1: define the outcome, and who is in the table

**What this does.** The outcome is "readmitted within 30 days". A person can have many stays in this data, so only each person's first stay is kept: otherwise the same person would sit in both the training part and the test part, and the model would look better than it is. People who died or went to hospice are left out because they cannot be readmitted.

**What it shows.** How many people remain and how common the outcome is. A rare outcome (here under one in ten) is why accuracy is a poor measure and why the next steps look at the cases the model catches or misses.

**What it does not show.** Whether "readmitted within 30 days" is the right outcome to care about, or whether being readmitted reflects the person's illness, the hospital's care, or what happened after discharge. The label comes from hospital records, and records can differ across groups for reasons that have nothing to do with health.

In [ ]:
first_stays = df.sort_values(ORDER_COLUMN).drop_duplicates(ID_COLUMN, keep='first')
kept = first_stays[~first_stays[EXCLUDE_COLUMN].isin(EXCLUDE_VALUES)].reset_index(drop=True)
y = (kept[LABEL] == POSITIVE).astype(int)

print(f'rows in the file: {len(df):,}')
print(f'people after keeping the first stay: {len(first_stays):,}')
print(f'people after leaving out those who cannot be readmitted: {len(kept):,}')
print(f'share with the outcome: {y.mean():.3f}')
print()
print('share of values missing, five most missing columns:')
print(kept.isna().mean().sort_values(ascending=False).head(5).round(3).to_string())

### Step 2: a simple model

**What this does.** It splits the people into a training part (70 percent) and a test part (30 percent), fits a plain logistic regression on the columns in `NUMERIC` and `CATEGORICAL`, and scores the test part. The group columns are not model inputs. Every number below comes from the test part, which the model has not seen. If you named `SCORE_COLUMN`, your own scores are used instead and nothing is trained.

**What it shows.** A stand-in for "the model the team built".

**What it does not show.** That a model without group columns is blind to groups. Other columns can stand in for them. And this model is simple on purpose: a better model would change the numbers, not the method.

In [ ]:
train_index, test_index = train_test_split(kept.index, test_size=0.3, stratify=y, random_state=SEED)

if SCORE_COLUMN is None:
    preprocess = ColumnTransformer([
        ('numbers', StandardScaler(), NUMERIC),
        ('categories', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ])
    inputs = kept[NUMERIC + CATEGORICAL].copy()
    inputs[CATEGORICAL] = inputs[CATEGORICAL].astype(str)  # codes such as 1 and 2 are labels here, not amounts
    model = make_pipeline(preprocess, LogisticRegression(max_iter=1000))
    model.fit(inputs.loc[train_index], y[train_index])
    score = pd.Series(model.predict_proba(inputs)[:, 1], index=kept.index)
else:
    score = kept[SCORE_COLUMN].astype(float)

test = pd.DataFrame({'y': y[test_index], 'score': score[test_index]})
test['flag'] = (test['score'] >= THRESHOLD).astype(int)

raw_groups = kept.loc[test_index, GROUP_COLUMNS].fillna('missing').astype(str)
GROUPINGS = {name: raw_groups[name] for name in GROUP_COLUMNS}
for columns in INTERSECTIONS:
    GROUPINGS[' x '.join(columns)] = raw_groups[columns].agg(' / '.join, axis=1)

print(f'people in the test set: {len(test):,}, of whom {test.y.sum():,} have the outcome')
print(f'share flagged at threshold {THRESHOLD}: {test.flag.mean():.3f}')

### Step 3: the overall result

**What this does.** It reports how the flagged group compares with the truth for everyone together:

- **flagged share**: the share of people whose score is at or above `THRESHOLD`, so the model flags them. Fairlearn calls this the selection rate.
- **recall** (also called sensitivity): out of the people who really were readmitted, the share the model flagged. This is the one that matters most when a missed case has a cost.
- **precision**: out of the people flagged, the share who really were readmitted. Low precision means many people are flagged who did not need it.
- **false positive rate**: out of the people who were not readmitted, the share flagged anyway.
- **AUC**: how often the model gives a higher score to a person who was readmitted than to one who was not, over all thresholds. 0.5 is a coin toss and 1 is perfect.
- **balanced accuracy**: the average of recall and the share of non-cases correctly left alone.

**What it shows.** The single numbers that usually go in a summary.

**What it does not show.** Anything about who gets these results. One number for everyone can be good because it is good for the majority and bad for a minority.

In [ ]:
def metrics_for(y_true, flag, score_values):
    """Recall, precision, false positive rate and AUC for one set of people."""
    positives = int(np.sum(y_true))
    out = {
        'people': len(y_true),
        'cases': positives,
        'flagged share': float(np.mean(flag)),
        'recall': recall_score(y_true, flag, zero_division=np.nan) if positives else np.nan,
        'precision': precision_score(y_true, flag, zero_division=np.nan) if np.sum(flag) else np.nan,
        'false positive rate': float(np.sum((flag == 1) & (y_true == 0)) / max(np.sum(y_true == 0), 1)),
        'AUC': roc_auc_score(y_true, score_values) if 0 < positives < len(y_true) else np.nan,
    }
    return out


overall = metrics_for(test.y.to_numpy(), test.flag.to_numpy(), test.score.to_numpy())
overall['balanced accuracy'] = balanced_accuracy_score(test.y, test.flag)
pd.Series(overall).round(3).to_frame('overall')

### Step 4: the same numbers for each group and intersection

**What this does.** It repeats Step 3 for every group of every column in `GROUP_COLUMNS`, and for every combination in `INTERSECTIONS` (for example, one gender within one race). Fairlearn's `MetricFrame` gives each group's size, flagged share and false positive rate. Recall, precision and AUC are computed group by group beside it with the same function as Step 3, so a group with no cases shows a blank instead of an error. A group is **reported** only if it has at least `MIN_GROUP_SIZE` people and at least `MIN_POSITIVES` true cases. Smaller groups stay in the table, marked, so you can see who is missing.

**What it shows.** Where the model works better or worse, and who is too small to say anything about.

**What it does not show.** Whether a gap is real. A group of 300 people with 30 cases gives a recall that could easily be ten points off. That is what Step 5 is for. It also does not show groups you never recorded.

In [ ]:
def table_for(feature):
    frame = MetricFrame(
        metrics={'people': count, 'flagged share': selection_rate, 'false positive rate': false_positive_rate},
        y_true=test.y, y_pred=test.flag, sensitive_features=feature)
    rows = {}
    for label, members in test.groupby(feature):
        rows[label] = metrics_for(members.y.to_numpy(), members.flag.to_numpy(), members.score.to_numpy())
    table = pd.DataFrame(rows).T
    for column in ['people', 'flagged share', 'false positive rate']:
        table[column] = frame.by_group[column]  # the flag-based numbers are MetricFrame's
    table['people'] = table['people'].astype(int)
    table['cases'] = table['cases'].astype(int)
    table['reported'] = (table['people'] >= MIN_GROUP_SIZE) & (table['cases'] >= MIN_POSITIVES)
    return table


tables = {name: table_for(feature) for name, feature in GROUPINGS.items()}
for name, table in tables.items():
    print(f'--- by {name} ---')
    print(table.round(3).to_string())
    print()

### Step 5: intervals, so a small group is not over-read

**What this does.** For each reported group it draws `N_BOOTSTRAP` new groups of the same size by picking people at random with replacement (scikit-learn's `resample`), recomputes recall, precision and AUC each time, and keeps the range that holds the middle 95 percent. This is a bootstrap interval. The width of the interval is the honest size of "we do not know exactly". Resamples that give no value for a metric are left out (for example, a resample with no flagged person has no precision). If more than one resample in ten gives no value, no interval is given for that metric, and Step 6 says `interval not available`.

**What it shows.** How much each group's number could move if the group were a slightly different sample. Two groups whose intervals overlap heavily cannot be told apart by this data.

**What it does not show.** It cannot repair a group that is not representative of the people you serve, and it says nothing about groups below the minimum size.

In [ ]:
def interval(values):
    clean = [v for v in values if not np.isnan(v)]
    return (np.quantile(clean, 0.025), np.quantile(clean, 0.975)) if len(clean) >= 0.9 * len(values) else (np.nan, np.nan)


interval_rows = []
for name, feature in GROUPINGS.items():
    for label, members in test.groupby(feature):
        if not tables[name].loc[label, 'reported']:
            continue
        y_g, flag_g, score_g = members.y.to_numpy(), members.flag.to_numpy(), members.score.to_numpy()
        draws = []
        for b in range(N_BOOTSTRAP):
            pick = resample(np.arange(len(members)), replace=True, random_state=SEED + b)
            draws.append(metrics_for(y_g[pick], flag_g[pick], score_g[pick]))
        draws = pd.DataFrame(draws)
        row = {'grouping': name, 'group': label, 'people': len(members), 'cases': int(y_g.sum())}
        for metric in ['recall', 'precision', 'AUC']:
            low, high = interval(draws[metric])
            row.update({metric: tables[name].loc[label, metric], f'{metric} low': low, f'{metric} high': high})
        interval_rows.append(row)

per_group = pd.DataFrame(interval_rows)
print(per_group.round(3).to_string(index=False))

### Step 6: does the overall result hide what happens inside groups?

**What this does.** Three checks.

1. **Group against overall.** For each reported group it compares the group's recall with the overall recall. If the whole 95 percent interval is below the overall value, the group is **below overall**: for that one group, chance alone is an unlikely explanation. If the whole interval is above, it is above overall. Otherwise the data **cannot tell**. Many groups are checked at once (30 in the example), so one or two can land outside by chance even with no real gap. A group near the edge can also change label with another seed or number of resamples. The same comparison is made for AUC and written to the per-group file as `AUC against overall`; read that column with check 2 in mind.
2. **Overall AUC against AUC inside groups.** Overall AUC can be higher than the AUC inside every group, because the model is partly telling groups apart (groups with a higher base rate get higher scores) rather than ranking people within a group. The check reports the overall value next to the lowest and highest values among the reported groups of the first column in `GROUP_COLUMNS`.
3. **Trend check.** For the yes/no factor you named, it compares the outcome rate of people with the factor and without it. It does this overall, and inside each group of the first column in `GROUP_COLUMNS` that meets the reporting rule (at least `MIN_GROUP_SIZE` people and `MIN_POSITIVES` cases). It then lists the groups where the direction is the opposite of the overall one. This check has no interval, so a flip in one group with a small difference can be chance. When the direction reverses inside most or all groups, the overall number is misleading: this is Simpson's paradox. A flip in one or two groups is a reason to look closer, not proof.

**What it shows.** Where "the model works" or "the factor matters" is only true on average.

**What it does not show.** Why a difference exists. It can come from the data, the world, or the model, and finding which is a separate piece of work.

In [ ]:
def against_overall(row, metric):
    if np.isnan(row[f'{metric} low']):
        return 'interval not available'
    if row[f'{metric} high'] < overall[metric]:
        return 'below overall'
    if row[f'{metric} low'] > overall[metric]:
        return 'above overall'
    return 'cannot tell'


per_group['recall against overall'] = per_group.apply(lambda row: against_overall(row, 'recall'), axis=1)
per_group['AUC against overall'] = per_group.apply(lambda row: against_overall(row, 'AUC'), axis=1)
print('Groups whose recall interval is entirely below the overall recall:')
print(per_group.loc[per_group['recall against overall'] == 'below overall', ['grouping', 'group', 'people', 'cases', 'recall', 'recall low', 'recall high']].round(3).to_string(index=False))
print()
print(per_group['recall against overall'].value_counts().to_string())

first_grouping = GROUP_COLUMNS[0]
single = tables[first_grouping]
print()
print(f'overall AUC {overall["AUC"]:.3f}; inside groups of {first_grouping}: lowest {single.loc[single.reported, "AUC"].min():.3f}, highest {single.loc[single.reported, "AUC"].max():.3f}')

factor = (kept[FACTOR_COLUMN] == FACTOR_VALUE).astype(int)
overall_effect = y[factor == 1].mean() - y[factor == 0].mean()
flips = []
for label, members in kept.fillna({first_grouping: 'missing'}).groupby(first_grouping):
    f, o = factor[members.index], y[members.index]
    if len(members) >= MIN_GROUP_SIZE and o.sum() >= MIN_POSITIVES and f.nunique() == 2:
        effect = o[f == 1].mean() - o[f == 0].mean()
        if np.sign(effect) != np.sign(overall_effect):
            flips.append((label, round(float(effect), 3)))
print(f'overall: outcome rate with {FACTOR_COLUMN} = {FACTOR_VALUE} minus without = {overall_effect:+.3f}')
print(f'groups of {first_grouping} where the direction is the opposite: {flips if flips else "none in this data"}')

In [ ]:
# A picture of Step 5: recall with its interval for every reported group. The line is the overall recall.
reported_groupings = list(per_group['grouping'].unique())
fig, axes = plt.subplots(1, len(reported_groupings), figsize=(4.2 * len(reported_groupings), 5.5), squeeze=False)
for axis, name in zip(axes[0], reported_groupings):
    part = per_group[per_group.grouping == name].reset_index(drop=True)
    axis.errorbar(part['recall'], range(len(part)),
                  xerr=[part['recall'] - part['recall low'], part['recall high'] - part['recall']], fmt='o', capsize=3)
    axis.axvline(overall['recall'], linestyle='--', color='grey')
    axis.set_yticks(range(len(part)))
    axis.set_yticklabels(part['group'], fontsize=7)
    axis.set_title(name, fontsize=9)
    axis.set_xlabel('recall, with 95% interval')
fig.tight_layout()

### Stop and decide: which gaps matter enough to act on

The notebook shows gaps and how uncertain they are. It cannot tell you which ones to act on.

Questions to ask in the room, including someone who knows the setting where the model will be used:

1. Look at the groups whose recall is below overall with the interval entirely below. Is a miss in that group more costly, because of who they are or where they are treated?
2. For groups where the data cannot tell, is it acceptable to wait, or is the group one you must not leave unexamined? What would it take (more data, a longer period) to know?
3. Which groups are missing from the table because they were too small? Did you decide to leave them out, or did the size rule decide for you?
4. How large a gap would you act on, and who decides? Write the number down before looking at the table next time.
5. If the overall number is good and one group is poor, would you still use the model, restrict it, or fix it first (see C-15 for options)?
6. This notebook leads with recall because a missed readmission has a cost. Is that the cost that matters most where you work, or do false alarms matter as much, for example when a flag takes staff time or leads to a follow-up the person did not need? Precision and the false positive rate for each group are in the Step 4 tables, and a gap there may matter to you even where recall is level.

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
GAP_SIZE_WE_ACT_ON = ''       # for example: 'recall more than 10 points below overall, interval entirely below'
GROUPS_WE_WILL_ACT_ON = ''    # which groups, and what you will do
GROUPS_WE_COULD_NOT_ASSESS = ''   # the groups too small to report, and your plan for them
WHO_DECIDES = ''              # the person or body who decides, and where it is written down

In [ ]:
EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Small tables of numbers, one row per group. They hold no row of the data.
per_group.round(4).to_csv(EVIDENCE_DIR / 'C-14_per_group.csv', index=False)
pd.Series(overall).round(4).to_frame('overall').to_csv(EVIDENCE_DIR / 'C-14_overall.csv')
fig.savefig(EVIDENCE_DIR / 'C-14_recall_by_group.png', dpi=100, bbox_inches='tight')

not_reported = sum(int((~t['reported']).sum()) for t in tables.values())
versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
below = per_group.loc[per_group['recall against overall'] == 'below overall', ['grouping', 'group', 'recall']].round(3)
summary = f"""# C-14 Disaggregated Performance Analysis: evidence summary

Attach this file, with the CSV and PNG files beside it, to the decision it supports (questions Q-IP-04, Q-IP-14, Q-IP-19). The per-group table is the one to copy into the model card's disaggregated results table. It holds numbers only, never the data.

## What was run
- Outcome: {LABEL} equal to {POSITIVE}; one row per person (first by {ORDER_COLUMN}); {len(kept):,} people, {len(test):,} in the test set
- Groups: {', '.join(GROUP_COLUMNS)}; intersections: {'; '.join(' x '.join(c) for c in INTERSECTIONS)}
- Decision threshold: {THRESHOLD}
- Reporting rule: at least {MIN_GROUP_SIZE} people and {MIN_POSITIVES} true cases; {not_reported} groups did not meet it and are not reported
- Bootstrap resamples: {N_BOOTSTRAP}
- Model: {'practice logistic regression' if SCORE_COLUMN is None else 'scores from the column ' + SCORE_COLUMN}

## Overall
{pd.Series(overall).round(3).to_string()}

## Groups whose recall is below overall, interval entirely below
{below.to_string(index=False) if len(below) else 'none'}

## Choices made in the stop-and-decide cell
- Gap size we act on: {GAP_SIZE_WE_ACT_ON or 'not written yet'}
- Groups we will act on: {GROUPS_WE_WILL_ACT_ON or 'not written yet'}
- Groups we could not assess: {GROUPS_WE_COULD_NOT_ASSESS or 'not written yet'}
- Who decides: {WHO_DECIDES or 'not written yet'}

## Files beside this summary
C-14_per_group.csv, C-14_overall.csv, C-14_recall_by_group.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
"""
(EVIDENCE_DIR / 'C-14_summary.md').write_text(summary, encoding='utf-8')
print(summary)

## What this does not show, and where to go next

- **It does not say the model is safe to use.** It describes performance on one test set from one source. Performance in your own setting, with your own people and your own workflow, has to be checked there.
- **It covers the groups in your file.** A group you did not record, or that was too small to report, is not examined, and the summary says how many were left out.
- **Intervals are not guarantees.** A 95 percent interval from a bootstrap assumes the test set is a fair sample of the people you care about. If it is not, the interval can sit in the wrong place, and its width will not show it. With 200 resamples, the ends of an interval also move a little from one seed to another.
- **Many groups are compared at once.** When many groups are each checked at 95 percent, one or two can fall below or above overall by chance even when there is no real gap. A single "below overall" among many is a reason to look closer, not a finding on its own.
- **A gap is not a cause.** Different results can come from the data, the label, the world or the model.
- **Outcome labels can be biased.** If readmission is recorded or acted on differently for different groups, a model can look accurate and still repeat that.

Where to go next: C-13 Subgroup Fairness Metrics (selection and error rate gaps between groups), C-15 Bias Mitigation Techniques (what to try when a gap is too large), C-16 Stratified Model Evaluation (one pooled model or one per group), and the model card template, whose disaggregated results table takes the numbers from this notebook's per-group CSV file.